In [ ]:
"""
Choropleth: Book Bans by State
Loads Banned_Books.csv, aggregates bans per state, builds Plotly choropleth,
saves to maps/initial_bans_by_state.html.
"""
from pathlib import Path
import pandas as pd
import plotly.express as px

# Paths: run from project root (geographic_dashboard_analysis) or from scripts/
PROJECT_ROOT = Path.cwd() if (Path.cwd() / "Banned_Books.csv").exists() else Path.cwd().parent
BANNED_CSV = PROJECT_ROOT / "Banned_Books.csv"
MAPS_DIR = PROJECT_ROOT / "maps"
OUTPUT_HTML = MAPS_DIR / "initial_bans_by_state.html"

# State full name -> two-letter abbreviation (Plotly USA-states expects abbreviations)
STATE_TO_ABBR = {
    "Alabama": "AL", "Alaska": "AK", "Arizona": "AZ", "Arkansas": "AR", "California": "CA",
    "Colorado": "CO", "Connecticut": "CT", "Delaware": "DE", "Florida": "FL", "Georgia": "GA",
    "Hawaii": "HI", "Idaho": "ID", "Illinois": "IL", "Indiana": "IN", "Iowa": "IA",
    "Kansas": "KS", "Kentucky": "KY", "Louisiana": "LA", "Maine": "ME", "Maryland": "MD",
    "Massachusetts": "MA", "Michigan": "MI", "Minnesota": "MN", "Mississippi": "MS",
    "Missouri": "MO", "Montana": "MT", "Nebraska": "NE", "Nevada": "NV", "New Hampshire": "NH",
    "New Jersey": "NJ", "New Mexico": "NM", "New York": "NY", "North Carolina": "NC",
    "North Dakota": "ND", "Ohio": "OH", "Oklahoma": "OK", "Oregon": "OR", "Pennsylvania": "PA",
    "Rhode Island": "RI", "South Carolina": "SC", "South Dakota": "SD", "Tennessee": "TN",
    "Texas": "TX", "Utah": "UT", "Vermont": "VT", "Virginia": "VA", "Washington": "WA",
    "West Virginia": "WV", "Wisconsin": "WI", "Wyoming": "WY", "District of Columbia": "DC",
}

# Load and aggregate
df = pd.read_csv(BANNED_CSV)
df = df.dropna(subset=["State"])
state_counts = df["State"].value_counts().reset_index()
state_counts.columns = ["State", "count"]
state_counts["state_abbr"] = state_counts["State"].map(STATE_TO_ABBR)
state_counts = state_counts.dropna(subset=["state_abbr"])

# Choropleth: more bans = darker
fig = px.choropleth(
    state_counts,
    locations="state_abbr",
    locationmode="USA-states",
    color="count",
    hover_name="State",
    hover_data={"count": ":,", "state_abbr": False},
    color_continuous_scale="Reds",
    scope="usa",
    title="Book Bans by State",
)
fig.update_layout(
    title_font_size=18,
    height=600,
    margin=dict(l=0, r=0, t=50, b=0),
    geo=dict(bgcolor="rgba(0,0,0,0)", lakecolor="lightblue"),
)

MAPS_DIR.mkdir(parents=True, exist_ok=True)
fig.write_html(OUTPUT_HTML)
print(f"Saved: {OUTPUT_HTML}")
fig.show()